### Load Dataset

In [ ]:
import pandas as pd
import torch


dfs = {}
pt_filenames = {
    # Preprocessed dataset path after running the training code at least once.
    "train": "../data/training.pt",
    "val": "../data/test.pt",
}

for split, filename in pt_filenames.items():
    data = torch.load(filename)
    dfs[split] = pd.DataFrame({
        "label": [x[1] for x in data],
    })



### Load Ensemble Logits

In [16]:


ensemble_logits = {}
pt_filenames = {
    # Ensemble logits path
    "train": "../teacher_logits/teacher_logits.training.pt",
    "val": "../teacher_logits/teacher_logits.test.pt",
}

for split, filename in pt_filenames.items():
    data = torch.load(filename)
    ensemble_logits[split] = data



### Evaluate

In [17]:
import numpy as np


def get_score(hits, counts, pflag=False):
    """ evaluation metric """
    # normal accuracy
    sp = hits[0] / (counts[0] + 1e-10) * 100
    # abnormal accuracy
    se = sum(hits[1:]) / (sum(counts[1:]) + 1e-10) * 100
    sc = (sp + se) / 2.0

    if pflag:
        # print("************* Metrics ******************")
        print("S_p: {:.4f}, S_e: {:.4f}, Score: {:.4f}".format(sp, se, sc))

    return sp, se, sc


def score_pred(col_name, df, n_cls = 4, pflag=True):
    counts = [0] * n_cls
    hits = [0] * n_cls
    for _, row in df.iterrows():
        hit = row[col_name] == row['label']
        counts[row['label']] += 1
        hits[row['label']] += hit
    return get_score(hits, counts, pflag=pflag)

def most_common_pred(logits):
    from collections import Counter
    counter = Counter(np.argmax(logits, axis=1))
    most_common_value, count = counter.most_common(1)[0]
    return most_common_value

import itertools


val_df = dfs['val']
val_ensemble_logits = ensemble_logits['val']

for k in range(1, len(val_ensemble_logits[0]) + 1):    # (iterate different thresholds)
    print(f"{k=}")
    preds_mean = []
    for idx, row in val_df.iterrows():
        logits = val_ensemble_logits[idx][:k]
        mean_logits = np.mean(logits, axis=0)
        pred_mean = mean_logits.argmax(-1)
        preds_mean.append(pred_mean)
    
    key = f'pred_mean_{k}'
    val_df[key] = preds_mean
    sp, se, sc = score_pred(key, val_df)


k=1
S_p: 81.3806, S_e: 44.2651, Score: 62.8229
k=2
S_p: 85.6238, S_e: 41.0365, Score: 63.3302
k=3
S_p: 85.7505, S_e: 41.8012, Score: 63.7758
k=4
S_p: 85.7505, S_e: 42.5658, Score: 64.1582
k=5
S_p: 85.1805, S_e: 43.5004, Score: 64.3405
k=6
S_p: 87.2704, S_e: 41.2065, Score: 64.2384
k=7
S_p: 87.7771, S_e: 40.6117, Score: 64.1944
k=8
S_p: 88.0937, S_e: 40.6117, Score: 64.3527
k=9
S_p: 88.1571, S_e: 40.8666, Score: 64.5118
k=10
S_p: 87.9037, S_e: 40.6117, Score: 64.2577
k=11
S_p: 88.2204, S_e: 40.6117, Score: 64.4161
k=12
S_p: 88.7270, S_e: 40.2719, Score: 64.4995
k=13
S_p: 89.0437, S_e: 41.4613, Score: 65.2525
k=14
S_p: 88.6004, S_e: 41.1215, Score: 64.8609
k=15
S_p: 88.2837, S_e: 41.2065, Score: 64.7451
k=16
S_p: 89.2970, S_e: 41.4613, Score: 65.3792
k=17
S_p: 89.2970, S_e: 41.6313, Score: 65.4641
k=18
S_p: 89.1704, S_e: 42.0561, Score: 65.6132
k=19
S_p: 89.2970, S_e: 41.9711, Score: 65.6341
k=20
S_p: 89.6137, S_e: 41.5463, Score: 65.5800
k=21
S_p: 89.5503, S_e: 41.5463, Score: 65.5483
k